# Unidad 29 — Ordenamiento rápido (quicksort)

Semana 4 · Clase 1 · Fuente: PDF parte 1, págs. 113–153

[Índice de la semana](../semana-04.ipynb) | [Anterior](u28-ordenamiento-por-fusion.ipynb) | [Siguiente](../clase-2-greedy-divide-y-venceras/u30-enfoque-greedy.ipynb)

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## Objetivos de la unidad

Según el PDF (pág. 114–115), al terminar la unidad se debe poder:

- Comprender el algoritmo de ordenamiento rápido y resolver problemas de ordenamiento con él.
- Explicar la diferencia entre el ordenamiento rápido y el ordenamiento por fusión.
- Explicar la complejidad temporal del quicksort.
- Implementar un ordenamiento rápido que divide una lista sin ordenar según un **pivote** y ordena cada parte por separado.
- Implementar una función de **partición** que divide la lista según el pivote.
- Entender que el ordenamiento rápido es también un método de "divide y vencerás" que usa una función recursiva, como el ordenamiento por fusión.

Palabras clave (pág. 116): problema de ordenamiento, ordenamiento rápido, pivote y partición.

Conceptos previos (pág. 115): operadores de comparación, llamadas recursivas sobre un problema dividido y notación Big O.

**Objetivo del docente para esta clase:** implementar el ordenamiento rápido con diferentes estrategias de selección de pivote.

**Referencia:** PDF parte 1, págs. 114–116

## Contexto

En la unidad anterior se estudió el ordenamiento por fusión, que necesita **memoria adicional del mismo tamaño que la lista dada**. El PDF plantea la pregunta de esta unidad: ¿se puede diseñar un algoritmo de ordenamiento eficiente **sin espacio de memoria adicional**? A este tipo de algoritmo se le llama **in situ** (en el mismo lugar): reordena los elementos dentro de la propia lista, usando solo intercambios.

La analogía del PDF son 7 canicas de hierro con el mismo tamaño y distinto peso, y una balanza que solo permite dos operaciones: comparar el peso de dos canicas e intercambiar la posición de dos canicas.

**Referencia:** PDF parte 1, págs. 118–119

## Misión: separar las canicas con una canica de referencia

**Problema (pág. 118):** dadas las canicas con pesos `40, 50, 10, 70, 60, 30, 20`, la canica roja es la de peso 40. Solo con intercambios, hay que dejar a la izquierda las más ligeras que la roja y a la derecha las más pesadas.

**Estrategia (pág. 119):** con la canica roja como referencia (el **pivote**) se busca, desde la izquierda, una canica más pesada que la roja y, desde la derecha, una más ligera, y se intercambian. El PDF pregunta cuántas comparaciones hacen falta; la respuesta se observa más abajo: del orden de una por canica, es decir, proporcional a `n`.

La función de partición que se estudia en la sección "Implementación" resuelve exactamente este problema. Aquí se usa una versión que además cuenta comparaciones.

In [2]:
def separar_canicas(pesos):
    """Mueve las canicas más ligeras que la primera a su izquierda y las más pesadas a su derecha."""
    S = list(pesos)
    pivote = S[0]
    izq, der = 1, len(S) - 1
    comparaciones = 0
    while izq <= der:
        while izq <= der:
            comparaciones += 1
            if S[izq] <= pivote:
                izq += 1
            else:
                break
        while izq <= der:
            comparaciones += 1
            if S[der] >= pivote:
                der -= 1
            else:
                break
        if izq < der:
            S[izq], S[der] = S[der], S[izq]
    S[0], S[der] = S[der], S[0]
    return S, der, comparaciones

canicas = [40, 50, 10, 70, 60, 30, 20]
resultado, posicion, comparaciones = separar_canicas(canicas)
print("Antes:   ", canicas)
print("Después: ", resultado)
print("Posición de la canica roja:", posicion)
print("Comparaciones realizadas:", comparaciones, "para", len(canicas), "canicas")

Antes:    [40, 50, 10, 70, 60, 30, 20]
Después:  [30, 20, 10, 40, 60, 70, 50]
Posición de la canica roja: 3
Comparaciones realizadas: 11 para 7 canicas


La disposición final no tiene por qué coincidir con la del dibujo del PDF (`20 10 30 40 50 70 60`): cualquier orden es válido mientras los más ligeros queden a la izquierda del pivote y los más pesados a la derecha. Que cada mitad quede ordenada internamente **no** forma parte de la misión.

**Clave:** partir una lista alrededor de un pivote cuesta del orden de `n` comparaciones y no usa memoria adicional.

**Referencia:** PDF parte 1, págs. 118–120

## Idea del ordenamiento rápido

El **ordenamiento rápido** (quicksort) ordena recursivamente una lista no ordenada dividiéndola en dos sublistas según el valor de un **pivote**: a la izquierda quedan los elementos menores que el pivote y a la derecha los mayores. Con la lista `[15, 10, 12, 20, 25, 13, 22]` el PDF toma siempre el primer elemento de cada sublista como pivote:

1. Pivote 15: `[13, 10, 12]`, `15`, `[25, 20, 22]`.
2. Se repite en cada sublista: pivote 13 a la izquierda y pivote 25 a la derecha.
3. Se sigue hasta que cada sublista tiene un solo elemento (o ninguno). Ese es el caso base de la recursión: una lista de un elemento ya está ordenada. Al terminar, toda la lista está ordenada.

La siguiente celda reproduce ese proceso con listas nuevas, solo para visualizar los niveles de la recursión.

**Ampliación (no está en el PDF):** esta versión crea listas nuevas en cada nivel (usa memoria adicional); la versión in situ del PDF aparece más adelante.

In [3]:
def quicksort_visual(lista, nivel=0):
    """Versión didáctica: muestra cómo se divide la lista en cada nivel de la recursión."""
    sangria = "    " * nivel
    if len(lista) <= 1:
        print(f"{sangria}{lista}  (caso base: ya está ordenada)")
        return lista
    pivote = lista[0]
    menores = [x for x in lista[1:] if x < pivote]
    mayores = [x for x in lista[1:] if x >= pivote]
    print(f"{sangria}{lista}  pivote={pivote}  menores={menores}  mayores={mayores}")
    return quicksort_visual(menores, nivel + 1) + [pivote] + quicksort_visual(mayores, nivel + 1)

print(quicksort_visual([15, 10, 12, 20, 25, 13, 22]))

[15, 10, 12, 20, 25, 13, 22]  pivote=15  menores=[10, 12, 13]  mayores=[20, 25, 22]
    [10, 12, 13]  pivote=10  menores=[]  mayores=[12, 13]
        []  (caso base: ya está ordenada)
        [12, 13]  pivote=12  menores=[]  mayores=[13]
            []  (caso base: ya está ordenada)
            [13]  (caso base: ya está ordenada)
    [20, 25, 22]  pivote=20  menores=[]  mayores=[25, 22]
        []  (caso base: ya está ordenada)
        [25, 22]  pivote=25  menores=[22]  mayores=[]
            [22]  (caso base: ya está ordenada)
            []  (caso base: ya está ordenada)
[10, 12, 13, 15, 20, 22, 25]


La diferencia con el ordenamiento por fusión (pág. 138): ambos usan "dividir y conquistar" con recursión, pero el ordenamiento rápido divide **según el pivote**, así que las dos sublistas pueden tener tamaños distintos. En la fusión la lista siempre se parte por la mitad y el trabajo se hace al combinar; en el rápido el trabajo se hace al dividir y no hay nada que combinar.

**Clave:** el ordenamiento rápido es divide y vencerás, recursivo, con el trabajo concentrado en la partición.

**Para el quiz:** la diferencia con la fusión es que el tamaño de las dos sublistas depende del pivote y puede ser desigual.

**Referencia:** PDF parte 1, págs. 122–126 y 138

## Partición con pivote (in situ)

Para no usar almacenamiento separado, el PDF particiona la lista **dentro de sí misma** con dos índices que se acercan desde los extremos (siempre excluyendo el pivote, que se queda en la primera posición durante el proceso):

1. De izquierda a derecha se busca el primer elemento **mayor** que el pivote. Con `[15, 10, 12, 20, 25, 13, 22]`: 10 y 12 son menores que 15 y se saltan; 20 es el primero mayor.
2. De derecha a izquierda se busca el primer elemento **menor** que el pivote: 22 es mayor y se salta; 13 es el primero menor.
3. Se **intercambian** el mayor encontrado (20) y el menor encontrado (13): queda `15 10 12 13 25 20 22`.
4. Se repite hasta que los índices se cruzan. Entonces la posición del índice derecho es la **posición del pivote**: se intercambian el pivote (primera posición) y ese elemento. Resultado: `13 10 12 15 25 20 22`, con los menores a la izquierda del 15 y los mayores a la derecha.

La función siguiente muestra esos pasos.

**Ampliación (no está en el PDF):** el código de traza es propio; los pasos y los valores son los del PDF.

In [4]:
def partir_con_traza(lista):
    """Parte la lista alrededor del primer elemento y explica cada paso."""
    S = list(lista)
    pivote = S[0]
    izq, der = 1, len(S) - 1
    print(f"Pivote = {pivote}, lista = {S}")
    while izq <= der:
        while izq <= der and S[izq] <= pivote:
            print(f"  izquierda: {S[izq]} <= {pivote}, avanza")
            izq += 1
        if izq <= der:
            print(f"  izquierda: {S[izq]} > {pivote}, se detiene en la posición {izq}")
        while izq <= der and S[der] >= pivote:
            print(f"  derecha: {S[der]} >= {pivote}, retrocede")
            der -= 1
        if izq <= der:
            print(f"  derecha: {S[der]} < {pivote}, se detiene en la posición {der}")
        if izq < der:
            S[izq], S[der] = S[der], S[izq]
            print(f"  intercambio: {S}")
    S[0], S[der] = S[der], S[0]
    print(f"El pivote se coloca en la posición {der}: {S}")
    return S, der

partir_con_traza([15, 10, 12, 20, 25, 13, 22])

Pivote = 15, lista = [15, 10, 12, 20, 25, 13, 22]
  izquierda: 10 <= 15, avanza
  izquierda: 12 <= 15, avanza
  izquierda: 20 > 15, se detiene en la posición 3
  derecha: 22 >= 15, retrocede
  derecha: 13 < 15, se detiene en la posición 5
  intercambio: [15, 10, 12, 13, 25, 20, 22]
  izquierda: 13 <= 15, avanza
  izquierda: 25 > 15, se detiene en la posición 4
  derecha: 20 >= 15, retrocede
  derecha: 25 >= 15, retrocede
El pivote se coloca en la posición 3: [13, 10, 12, 15, 25, 20, 22]


([13, 10, 12, 15, 25, 20, 22], 3)

**Clave:** al terminar la partición el pivote queda en su **posición definitiva**; ya no hay que moverlo en las llamadas recursivas.

**Error común:** creer que la partición ordena las dos mitades. Solo garantiza "menores a la izquierda, mayores a la derecha"; cada mitad sigue desordenada hasta que se la ordena recursivamente.

**Para el quiz:** tras `partition1` el pivote está en su posición final, con los menores a su izquierda y los mayores a su derecha.

**Referencia:** PDF parte 1, págs. 127–134

## Elección del pivote y el peor caso

El PDF usa primero la estrategia de elegir **el primer elemento** como pivote y pregunta qué ocurre con una lista **ya ordenada**, como `[10, 12, 13, 15, 20, 22, 25]`. Cada vez el primer elemento es el más pequeño, la partición no divide nada (queda una sublista vacía y otra con `n - 1` elementos) y hay que repetir el proceso `N` veces (pág. 136).

La siguiente celda lo comprueba contando el tamaño de la sublista mayor en cada partición.

In [5]:
def niveles_primer_pivote(lista):
    """Cuenta cuántas particiones hace el ordenamiento rápido con primer pivote (versión con listas)."""
    particiones = 0
    pendientes = [list(lista)]
    while pendientes:
        actual = pendientes.pop()
        if len(actual) <= 1:
            continue
        particiones += 1
        pivote = actual[0]
        pendientes.append([x for x in actual[1:] if x < pivote])
        pendientes.append([x for x in actual[1:] if x >= pivote])
    return particiones

ordenada = [10, 12, 13, 15, 20, 22, 25]
print("Lista ya ordenada:", ordenada)
print("Particiones con primer pivote:", niveles_primer_pivote(ordenada), "(n - 1 = 6)")
print("Particiones en la lista desordenada del ejemplo:", niveles_primer_pivote([15, 10, 12, 20, 25, 13, 22]))

Lista ya ordenada: [10, 12, 13, 15, 20, 22, 25]
Particiones con primer pivote: 6 (n - 1 = 6)
Particiones en la lista desordenada del ejemplo: 5


**Clave:** elegir siempre el primer elemento como pivote es una mala idea cuando la entrada ya está (casi) ordenada, porque la partición queda totalmente desbalanceada.

**Error común:** pensar que el peor caso necesita una lista "muy desordenada". Con el primer elemento como pivote, la lista ya ordenada (y también la inversa, como se ve más adelante) es el peor caso.

**Referencia:** PDF parte 1, págs. 135–136

## Implementación con primer pivote: `quicksort1` y `partition1`

Código del PDF (págs. 138 y 140). Para ordenar la lista `S` entera se llama `quicksort1(S, 0, len(S) - 1)`; `low` y `high` son los índices del tramo que se está ordenando.

- `quicksort1`: si `low < high`, particiona con `partition1`, y ordena recursivamente el tramo izquierdo (`low` a `pivotpoint - 1`) y el derecho (`pivotpoint + 1` a `high`). Si `low >= high` el tramo tiene 0 o 1 elementos y ya está ordenado: no hace nada.
- `partition1`: toma `S[low]` como pivote, mueve `left` hacia la derecha mientras los elementos sean menores o iguales al pivote y `right` hacia la izquierda mientras sean mayores o iguales, intercambia `S[left]` y `S[right]` cuando ambos se detienen, y al cruzarse coloca el pivote en `right`.

Ambas funciones incluyen los `print(S)` del PDF, que muestran el estado de la lista en cada paso.

In [6]:
def quicksort1(S, low, high):
    if low < high:
        print(S)
        pivotpoint = partition1(S, low, high)
        quicksort1(S, low, pivotpoint - 1)
        quicksort1(S, pivotpoint + 1, high)

def partition1(S, low, high):
    pivot = S[low]
    left, right = low + 1, high
    while left < right:
        print(S)
        while left <= right and S[left] <= pivot:
            left += 1
        while left <= right and S[right] >= pivot:
            right -= 1
        if left < right:
            S[left], S[right] = S[right], S[left]
    pivotpoint = right
    S[low], S[pivotpoint] = S[pivotpoint], S[low]
    return pivotpoint

**Nota sobre el PDF:** la explicación de las líneas 1 a 3 de `quicksort1` dice que "si low es igual o menor que high, el ordenamiento puede terminar". Es una traducción confusa: el código es `if low < high:` y el tramo **solo se ordena** mientras `low < high`; cuando `low >= high` termina sin hacer nada. Además, el PDF dice que `partition1` ordena "de menor a mayor"; en realidad solo separa menores y mayores respecto al pivote.

Ejecución con la lista del PDF (pág. 143). El resultado coincide línea por línea con la salida mostrada en el PDF.

In [7]:
S = [15, 10, 12, 20, 25, 13, 22]
quicksort1(S, 0, len(S) - 1)
print(S)

[15, 10, 12, 20, 25, 13, 22]
[15, 10, 12, 20, 25, 13, 22]
[15, 10, 12, 13, 25, 20, 22]
[13, 10, 12, 15, 25, 20, 22]
[13, 10, 12, 15, 25, 20, 22]
[12, 10, 13, 15, 25, 20, 22]
[10, 12, 13, 15, 25, 20, 22]
[10, 12, 13, 15, 25, 20, 22]
[10, 12, 13, 15, 22, 20, 25]
[10, 12, 13, 15, 20, 22, 25]


Cada línea impresa es el estado de la lista al entrar a `quicksort1` o a una iteración de `partition1`; por eso algunas líneas se repiten. La última línea es la lista ya ordenada.

**Prueba sorpresa del PDF (pág. 149, Q1):** *Dada la lista `S = [15, 10, 12, 20, 25, 13, 22]`, escriba la salida tras ejecutar `partition1(S, 0, len(S) - 1)` y luego `print(S)`.* La respuesta es la lista con el pivote 15 ya colocado:

In [8]:
S = [15, 10, 12, 20, 25, 13, 22]
partition1(S, 0, len(S) - 1)   # imprime los estados intermedios de la partición
print(S)                        # lista tras la partición (salida pedida en la prueba sorpresa)
verificar("prueba sorpresa Q1", S, [13, 10, 12, 15, 25, 20, 22])

[15, 10, 12, 20, 25, 13, 22]
[15, 10, 12, 13, 25, 20, 22]
[13, 10, 12, 15, 25, 20, 22]
Correcto: prueba sorpresa Q1


Peor caso (pág. 144): si la lista llega en **orden descendente**, el primer elemento es el mayor, siempre queda a la derecha y en cada llamada recursiva se separan exactamente `N - 1` elementos.

In [9]:
S = [25, 22, 20, 15, 13, 12, 10]
quicksort1(S, 0, len(S) - 1)
print(S)

[25, 22, 20, 15, 13, 12, 10]
[25, 22, 20, 15, 13, 12, 10]
[10, 22, 20, 15, 13, 12, 25]
[10, 22, 20, 15, 13, 12, 25]
[10, 22, 20, 15, 13, 12, 25]
[10, 22, 20, 15, 13, 12, 25]
[10, 12, 20, 15, 13, 22, 25]
[10, 12, 20, 15, 13, 22, 25]
[10, 12, 20, 15, 13, 22, 25]
[10, 12, 20, 15, 13, 22, 25]
[10, 12, 13, 15, 20, 22, 25]
[10, 12, 15, 13, 20, 22, 25]


**Nota sobre el PDF (error en `partition1`):** el código de la pág. 140 usa `while left < right:` como condición del bucle externo. Con esa condición, cuando el tramo solo deja **un** candidato (`left == right` desde el principio, por ejemplo un tramo de dos elementos), el bucle no se ejecuta, no se compara ese elemento con el pivote y la función intercambia el pivote con él aunque sea mayor. Los ejemplos del PDF salen bien por casualidad, pero otras listas quedan mal ordenadas. La corrección es `while left <= right:`. Se comprueba a continuación.

In [10]:
for lista in ([1, 2, 3], [2, 3, 1]):
    S = list(lista)
    import contextlib, io
    with contextlib.redirect_stdout(io.StringIO()):   # se silencian los print del PDF
        quicksort1(S, 0, len(S) - 1)
    print(f"{lista} -> {S}   ordenada correctamente: {S == sorted(lista)}")

# Frecuencia del fallo con listas aleatorias pequeñas (semilla fija)
import random
generador = random.Random(1)
fallos = 0
for _ in range(1000):
    lista = generador.sample(range(20), generador.randint(2, 9))
    S = list(lista)
    with contextlib.redirect_stdout(io.StringIO()):
        quicksort1(S, 0, len(S) - 1)
    fallos += S != sorted(lista)
print("Listas mal ordenadas por el código del PDF:", fallos, "de 1000")

[1, 2, 3] -> [1, 3, 2]   ordenada correctamente: False
[2, 3, 1] -> [1, 2, 3]   ordenada correctamente: True
Listas mal ordenadas por el código del PDF: 515 de 1000


Versión corregida (sin los `print`, lista para usar en el resto del notebook). Solo cambia la condición del bucle externo:

In [11]:
def partition1_corregida(S, low, high):
    pivot = S[low]
    left, right = low + 1, high
    while left <= right:            # corrección: el PDF usa "left < right"
        while left <= right and S[left] <= pivot:
            left += 1
        while left <= right and S[right] >= pivot:
            right -= 1
        if left < right:
            S[left], S[right] = S[right], S[left]
    pivotpoint = right
    S[low], S[pivotpoint] = S[pivotpoint], S[low]
    return pivotpoint

def quicksort1_corregido(S, low, high):
    if low < high:
        pivotpoint = partition1_corregida(S, low, high)
        quicksort1_corregido(S, low, pivotpoint - 1)
        quicksort1_corregido(S, pivotpoint + 1, high)

generador = random.Random(1)
fallos = 0
for _ in range(1000):
    lista = generador.sample(range(20), generador.randint(2, 9))
    S = list(lista)
    quicksort1_corregido(S, 0, len(S) - 1)
    fallos += S != sorted(lista)
print("Listas mal ordenadas con la versión corregida:", fallos, "de 1000")

Listas mal ordenadas con la versión corregida: 0 de 1000


**Clave:** `partition1` devuelve `pivotpoint`, la posición final del pivote; `quicksort1` solo recurre sobre los dos tramos que quedan a cada lado.

**Error común:** llamar a `quicksort1(S, 0, len(S))` (sin el `- 1`): `high` es un índice válido, no la longitud. Y confiar en el `partition1` del PDF tal cual: tiene el error descrito arriba.

**Para el quiz:** el peor caso del primer pivote es la lista ordenada (ascendente) o la inversa (descendente); en ambos las divisiones son de `N - 1` elementos.

**Referencia:** PDF parte 1, págs. 138–144 y 149

## Pivote aleatorio: `quicksort2` y `partition2`

Para evitar que una entrada concreta (ordenada o inversa) sea siempre la peor, `quicksort2` es idéntica a `quicksort1` pero llama a `partition2`, que **elige una posición aleatoria** `rand` entre `low` y `high` e intercambia `S[low]` con `S[rand]`; así el pivote aleatorio queda en la primera posición y el resto del algoritmo es parecido (págs. 145–146).

En esta versión del PDF los bucles internos usan los límites `left < high` y `right > low`, en vez de `left <= right`.

Se fija la semilla para que la salida de este notebook sea reproducible; en una ejecución real los pivotes elegidos cambian cada vez.

In [12]:
import random
from random import randint

def quicksort2(S, low, high):
    if low < high:
        pivotpoint = partition2(S, low, high)
        quicksort2(S, low, pivotpoint - 1)
        quicksort2(S, pivotpoint + 1, high)

def partition2(S, low, high):
    rand = randint(low, high)
    S[low], S[rand] = S[rand], S[low]
    pivot, left, right = S[low], low, high
    print(S, left, right, "pivot = ", pivot)
    while left < right:
        while left < high and S[left] <= pivot:
            left += 1
        while right > low and pivot <= S[right]:
            right -= 1
        if left < right:
            S[left], S[right] = S[right], S[left]
    S[low], S[right] = S[right], S[low]
    return right

random.seed(42)   # semilla fija: la salida es reproducible
S = [25, 22, 20, 15, 13, 12, 10]
quicksort2(S, 0, len(S) - 1)
print(S)

[12, 22, 20, 15, 13, 25, 10] 0 6 pivot =  12
[10, 12, 20, 15, 13, 25, 22] 2 6 pivot =  20
[10, 12, 13, 15, 20, 25, 22] 2 3 pivot =  13
[10, 12, 13, 15, 20, 22, 25] 5 6 pivot =  22
[10, 12, 13, 15, 20, 22, 25]


**Nota sobre el PDF:** la salida de la pág. 147 corresponde a una ejecución concreta sin semilla (pivotes 12, 25, 22 y 15). Con otra semilla los pivotes y las líneas impresas serán distintos; lo que no cambia es el resultado final ordenado. Cada línea impresa muestra la lista **tras** el intercambio inicial, los índices `left` y `right` (el tramo actual) y el pivote.

Lo importante que remarca el PDF: con pivote aleatorio la lista descendente **deja de ser el peor caso** de forma sistemática.

**Clave:** el pivote aleatorio no elimina el peor caso, pero lo hace improbable para cualquier entrada concreta.

**Error común:** creer que "aleatorio" garantiza velocidad. Con mala suerte puede elegirse un pivote extremo muchas veces seguidas, aunque la probabilidad es muy baja.

**Nota sobre el PDF:** `partition2` no tiene el error de `partition1`: sus bucles internos están acotados por `high` y `low` y comienza con `left = low`, por lo que funciona con cualquier lista (se comprobó con listas aleatorias, con y sin repetidos).

**Para el quiz:** `quicksort2` solo cambia la forma de elegir el pivote (`partition2`); la estructura recursiva es la misma que en `quicksort1`.

**Referencia:** PDF parte 1, págs. 145–147

## Estrategias de selección de pivote y su efecto en el rendimiento

El PDF compara dos estrategias: primer elemento y elemento aleatorio. Para cumplir el objetivo del docente se generalizan en una sola implementación donde la estrategia es un parámetro, y se agregan otras tres habituales. Todas usan el mismo esquema de partición que `partition1`: la estrategia solo decide **qué elemento se lleva a la primera posición** antes de particionar.

**Ampliación (no está en el PDF):** las estrategias último, medio y mediana de tres, y las mediciones. Se cuentan comparaciones y la profundidad máxima de la recursión, que son medidas deterministas; los tiempos son orientativos.

In [13]:
import random
import time

class Contador:
    """Acumula comparaciones con el pivote y la profundidad máxima de la recursión."""
    def __init__(self):
        self.comparaciones = 0
        self.profundidad = 0

def elegir_primero(S, low, high):
    return low

def elegir_ultimo(S, low, high):
    return high

def elegir_medio(S, low, high):
    return (low + high) // 2

def elegir_aleatorio(S, low, high):
    return random.randint(low, high)

def elegir_mediana_de_tres(S, low, high):
    medio = (low + high) // 2
    candidatos = sorted([(S[low], low), (S[medio], medio), (S[high], high)])
    return candidatos[1][1]

ESTRATEGIAS = {
    "primero": elegir_primero,
    "ultimo": elegir_ultimo,
    "medio": elegir_medio,
    "aleatorio": elegir_aleatorio,
    "mediana de tres": elegir_mediana_de_tres,
}

def particionar(S, low, high, elegir, cont):
    """Partición del PDF (partition1) con estrategia de pivote intercambiable."""
    idx = elegir(S, low, high)
    S[low], S[idx] = S[idx], S[low]
    pivot = S[low]
    left, right = low + 1, high
    while left <= right:
        while left <= right:
            cont.comparaciones += 1
            if S[left] <= pivot:
                left += 1
            else:
                break
        while left <= right:
            cont.comparaciones += 1
            if S[right] >= pivot:
                right -= 1
            else:
                break
        if left < right:
            S[left], S[right] = S[right], S[left]
    S[low], S[right] = S[right], S[low]
    return right

def quicksort_estrategia(S, low, high, elegir, cont, profundidad=1):
    if low < high:
        cont.profundidad = max(cont.profundidad, profundidad)
        pp = particionar(S, low, high, elegir, cont)
        quicksort_estrategia(S, low, pp - 1, elegir, cont, profundidad + 1)
        quicksort_estrategia(S, pp + 1, high, elegir, cont, profundidad + 1)

def ordenar(lista, elegir):
    S = list(lista)
    cont = Contador()
    quicksort_estrategia(S, 0, len(S) - 1, elegir, cont)
    return S, cont

# Comprobación: todas las estrategias ordenan bien
random.seed(7)
datos = [15, 10, 12, 20, 25, 13, 22, 10, 15]
for nombre, f in ESTRATEGIAS.items():
    resultado, _ = ordenar(datos, f)
    verificar(f"estrategia {nombre} ordena", resultado, sorted(datos))

Correcto: estrategia primero ordena
Correcto: estrategia ultimo ordena
Correcto: estrategia medio ordena
Correcto: estrategia aleatorio ordena
Correcto: estrategia mediana de tres ordena


La siguiente tabla compara las estrategias sobre cuatro tipos de entrada con `n = 300` elementos (se usa un tamaño moderado para que el peor caso, con profundidad `n`, no agote el límite de recursión de Python). Las comparaciones y la profundidad no dependen de la velocidad de la máquina.

In [14]:
n = 300
generador = random.Random(2024)
entradas = {
    "ascendente": list(range(n)),
    "descendente": list(range(n, 0, -1)),
    "aleatoria": generador.sample(range(n), n),
    "todos iguales": [5] * n,
}

random.seed(2024)   # semilla fija para la estrategia aleatoria
print(f"n = {n}. Cada celda: comparaciones / profundidad máxima de recursión")
print(f"{'estrategia':<16}" + "".join(f"{nombre:>22}" for nombre in entradas))
for nombre_est, f in ESTRATEGIAS.items():
    fila = f"{nombre_est:<16}"
    for nombre_ent, datos in entradas.items():
        resultado, cont = ordenar(datos, f)
        assert resultado == sorted(datos)
        fila += f"{cont.comparaciones:>14} / {cont.profundidad:<5}"
    print(fila)
print("Referencia: n*(n-1)/2 =", n * (n - 1) // 2, " y n*log2(n) aprox.", round(n * 8.23))

n = 300. Cada celda: comparaciones / profundidad máxima de recursión
estrategia                  ascendente           descendente             aleatoria         todos iguales
primero                  45149 / 299           44999 / 299            4259 / 20            44850 / 299  
ultimo                   44850 / 299           45000 / 299            3528 / 16            44850 / 299  
medio                     2070 / 8              2448 / 10             3817 / 19            44850 / 299  
aleatorio                 2681 / 18             3062 / 17             3621 / 16            44850 / 299  
mediana de tres           2070 / 8             11921 / 76             3325 / 14            44850 / 299  
Referencia: n*(n-1)/2 = 44850  y n*log2(n) aprox. 2469


Cómo leer la tabla:

- **Primero** y **último** son los peores con entradas ordenadas o inversas: del orden de `n²/2` comparaciones y profundidad `n` (cada partición separa un solo elemento).
- **Medio** maneja bien las entradas ascendente y descendente (comparaciones del orden de `n log n` y profundidad logarítmica). **Mediana de tres** es excelente con la ascendente; con la descendente hace bastante más trabajo que el medio, aunque muy lejos del cuadrático, porque el intercambio que lleva el pivote a la primera posición altera el orden del tramo. Además existen entradas construidas a propósito que degradan tanto al medio como a la mediana de tres.
- **Aleatorio** no depende de la forma de la entrada: su rendimiento es parecido en ascendente, descendente y aleatoria.
- **Todos iguales** degrada **todas** las estrategias con este esquema de partición: como `S[left] <= pivot` y `S[right] >= pivot` avanzan sobre los iguales, el pivote queda siempre en un extremo.

**Error común:** suponer que una estrategia de pivote es mejor "en general". Cada una tiene entradas desfavorables; la elección depende de qué se sabe de los datos.

Ahora los tiempos, solo con datos aleatorios y con el tamaño `n = 20000` (los datos ordenados degradarían a la recursión profunda con primero/último). **Los tiempos son orientativos**: dependen de la máquina y de la carga del equipo; la semilla fija solo hace repetibles los datos, no el reloj.

In [15]:
n = 20000
datos = random.Random(11).sample(range(n * 10), n)

def mejor_tiempo(funcion, repeticiones=3):
    mejores = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion()
        mejores.append(time.perf_counter() - inicio)
    return min(mejores)

print(f"Tiempos orientativos con n = {n} datos aleatorios (semilla fija en los datos)")
for nombre_est, f in ESTRATEGIAS.items():
    random.seed(1)
    t = mejor_tiempo(lambda: ordenar(datos, f))
    _, cont = ordenar(datos, f)
    print(f"{nombre_est:<16} {t * 1000:8.1f} ms   comparaciones={cont.comparaciones:>7}   profundidad={cont.profundidad}")
t = mejor_tiempo(lambda: sorted(datos))
print(f"{'sorted() de Python':<16} {t * 1000:8.1f} ms   (referencia, implementado en C)")

Tiempos orientativos con n = 20000 datos aleatorios (semilla fija en los datos)
primero              14.6 ms   comparaciones= 449174   profundidad=30
ultimo               14.5 ms   comparaciones= 445542   profundidad=30


medio                14.8 ms   comparaciones= 453557   profundidad=36

aleatorio            16.7 ms   comparaciones= 449044   profundidad=31


mediana de tres      15.2 ms   comparaciones= 403434   profundidad=22
sorted() de Python      1.5 ms   (referencia, implementado en C)


**Clave:** con datos aleatorios las cinco estrategias hacen un número parecido de comparaciones (del orden de `n log n`); las diferencias aparecen con entradas ordenadas o con muchos valores repetidos.

**Para el quiz:** la estrategia de pivote decide el equilibrio de las particiones, y el equilibrio decide el rendimiento: particiones equilibradas dan `O(n log n)` y particiones de `n - 1` elementos dan `O(n²)`.

**Ampliación (no está en el PDF):** complejidad temporal. El PDF solo muestra que en el peor caso hay `N` divisiones de `n - 1` elementos. Resumen habitual: cada nivel de la recursión recorre del orden de `n` elementos; con particiones equilibradas hay unos `log2(n)` niveles, de modo que el caso promedio y el mejor son `O(n log n)`; con particiones desbalanceadas hay `n` niveles y el peor caso es `O(n²)`.

## Comparación de memoria: in situ frente a fusión

El PDF motiva el ordenamiento rápido porque la fusión usa memoria adicional del tamaño de la lista (pág. 118). Se mide a continuación con `tracemalloc` el **pico de memoria adicional** durante la ordenación de la misma lista con tres implementaciones: fusión con listas nuevas (como en la unidad 28), ordenamiento rápido con listas nuevas (`quicksort_visual` sin imprimir) y ordenamiento rápido in situ (el del PDF).

**Ampliación (no está en el PDF):** toda esta medición. `tracemalloc` cuenta los objetos que Python reserva; no cuenta los marcos de la pila de llamadas, así que además se informa la profundidad máxima de recursión. Los valores de bytes son orientativos (dependen de la versión de Python), pero el orden de magnitud sí es significativo.

In [16]:
import tracemalloc

def fusion_con_listas(S):
    if len(S) <= 1:
        return S[:]
    medio = len(S) // 2
    izq, der = fusion_con_listas(S[:medio]), fusion_con_listas(S[medio:])
    resultado, i, j = [], 0, 0
    while i < len(izq) and j < len(der):
        if izq[i] <= der[j]:
            resultado.append(izq[i]); i += 1
        else:
            resultado.append(der[j]); j += 1
    return resultado + izq[i:] + der[j:]

def quicksort_con_listas(S):
    if len(S) <= 1:
        return S[:]
    pivote = S[0]
    menores = [x for x in S[1:] if x < pivote]
    mayores = [x for x in S[1:] if x >= pivote]
    return quicksort_con_listas(menores) + [pivote] + quicksort_con_listas(mayores)

def quicksort_in_situ(S):
    cont = Contador()
    quicksort_estrategia(S, 0, len(S) - 1, elegir_medio, cont)
    return S

def pico_de_memoria(funcion, datos):
    """Devuelve (resultado, bytes de pico adicionales) durante funcion(datos)."""
    tracemalloc.start()
    base = tracemalloc.get_traced_memory()[0]
    resultado = funcion(datos)
    pico = tracemalloc.get_traced_memory()[1] - base
    tracemalloc.stop()
    return resultado, pico

n = 20000
original = random.Random(5).sample(range(n * 10), n)
tamano_lista = len(original) * 8   # bytes de los punteros de una lista de ese tamaño

print(f"n = {n}; una lista de ese tamaño ocupa aprox. {tamano_lista / 1024:.0f} KiB")
for nombre, f, copiar in [
    ("fusión con listas", fusion_con_listas, False),
    ("rápido con listas", quicksort_con_listas, False),
    ("rápido in situ", quicksort_in_situ, True),
]:
    copia = list(original)
    resultado, pico = pico_de_memoria(f, copia)
    assert resultado == sorted(original)
    print(f"{nombre:<20} pico adicional aprox. {pico / 1024:9.1f} KiB")

n = 20000; una lista de ese tamaño ocupa aprox. 156 KiB
fusión con listas    pico adicional aprox.     639.6 KiB


rápido con listas    pico adicional aprox.     674.0 KiB


rápido in situ       pico adicional aprox.       3.2 KiB


Lectura de la medición: las dos versiones con listas nuevas reservan memoria del orden de la lista completa (o varias veces ese tamaño por las copias intermedias), mientras que la versión in situ solo intercambia elementos dentro de la lista original y reserva casi nada de objetos nuevos.

Esto no significa que el ordenamiento rápido use "cero" memoria: la recursión usa pila, una por cada nivel. Con particiones equilibradas son unos `log2(n)` niveles; en el peor caso, `n` niveles.

**Ampliación (no está en el PDF):** una forma habitual de limitar la pila a `O(log n)` incluso con pivotes malos es recurrir primero sobre la sublista **más pequeña** y repetir el bucle sobre la más grande. No se implementa aquí.

**Clave:** ventaja del rápido in situ sobre la fusión: no necesita una lista auxiliar del tamaño de la entrada.

**Para el quiz:** el ordenamiento por fusión usa memoria adicional proporcional a `n`; el rápido in situ no usa lista auxiliar (solo la pila de la recursión).

**Referencia:** PDF parte 1, pág. 118 (motivación); la medición es una ampliación.

## Proyecto de la unidad: el k-ésimo elemento más grande

**Problema (Programación en pareja, pág. 153, Q1):** escribir un algoritmo que encuentre el **k-ésimo elemento más grande** dados `N` elementos desordenados. Resolverlo con los dos métodos que sugiere el PDF y analizar cuál es más eficiente:

1. Ordenar la lista con una función de ordenamiento y devolver el elemento k-ésimo.
2. Usar la función `partition()` para llamar de forma recursiva hasta que el pivote sea el elemento k-ésimo.

**Plan de programación:**

1. El k-ésimo mayor de una lista de `n` elementos (`k = 1` es el máximo) ocupa el índice `n - k` en la lista ordenada ascendentemente.
2. Método 1: ordenar y leer ese índice.
3. Método 2 (selección rápida): particionar; si el pivote cae en el índice objetivo, es la respuesta; si cae antes, seguir solo con el tramo derecho; si cae después, solo con el izquierdo. A diferencia del ordenamiento rápido, **solo se recurre en un lado**.
4. Comparar el trabajo realizado (comparaciones) por cada método.

**Ampliación (no está en el PDF):** el código del proyecto y la comparación son propios; los dos métodos sí vienen del PDF.

In [17]:
def kmayor_ordenando(lista, k):
    """Método 1: ordena y devuelve el k-ésimo mayor."""
    if not 1 <= k <= len(lista):
        raise ValueError("k debe estar entre 1 y el tamaño de la lista")
    return sorted(lista)[len(lista) - k]

def kmayor_particion(lista, k, elegir=elegir_primero, cont=None):
    """Método 2: reparte con la partición y sigue solo por el lado donde está el índice objetivo."""
    if not 1 <= k <= len(lista):
        raise ValueError("k debe estar entre 1 y el tamaño de la lista")
    cont = cont if cont is not None else Contador()
    S = list(lista)                 # copia: no modifica la lista del usuario
    objetivo = len(S) - k
    low, high = 0, len(S) - 1
    while low <= high:
        pp = particionar(S, low, high, elegir, cont)
        if pp == objetivo:
            return S[pp]
        if pp < objetivo:
            low = pp + 1            # el objetivo está en el lado derecho
        else:
            high = pp - 1           # el objetivo está en el lado izquierdo

datos = [15, 10, 12, 20, 25, 13, 22]
for k in (1, 2, 3, 7):
    a = kmayor_ordenando(datos, k)
    b = kmayor_particion(datos, k)
    print(f"k = {k}: ordenando -> {a}, partición -> {b}")
    verificar(f"ambos métodos coinciden para k={k}", a, b)

try:
    kmayor_particion(datos, 0)
except ValueError as error:
    print("Error esperado:", error)

k = 1: ordenando -> 25, partición -> 25
Correcto: ambos métodos coinciden para k=1
k = 2: ordenando -> 22, partición -> 22
Correcto: ambos métodos coinciden para k=2
k = 3: ordenando -> 20, partición -> 20
Correcto: ambos métodos coinciden para k=3
k = 7: ordenando -> 10, partición -> 10
Correcto: ambos métodos coinciden para k=7
Error esperado: k debe estar entre 1 y el tamaño de la lista


**Análisis de eficiencia (pregunta del PDF).** Se cuentan las comparaciones del método de partición (con pivote aleatorio, semilla fija) y se comparan con las del ordenamiento completo con la misma estrategia. Tamaños y semillas fijos; las comparaciones son deterministas, a diferencia del tiempo.

In [18]:
def comparaciones_ordenando(lista, elegir):
    _, cont = ordenar(lista, elegir)
    return cont.comparaciones

n = 5000
datos = random.Random(99).sample(range(n * 10), n)
k = 100   # el centésimo mayor

random.seed(3)
c1 = comparaciones_ordenando(datos, elegir_aleatorio)
random.seed(3)
cont = Contador()
valor = kmayor_particion(datos, k, elegir_aleatorio, cont)
verificar("el valor coincide con ordenar", valor, kmayor_ordenando(datos, k))
print(f"n = {n}, k = {k}")
print(f"Ordenar todo con rápido:      {c1} comparaciones")
print(f"Selección con partición:      {cont.comparaciones} comparaciones")

Correcto: el valor coincide con ordenar
n = 5000, k = 100
Ordenar todo con rápido:      93685 comparaciones
Selección con partición:      21260 comparaciones


**Conclusión:** la selección con partición hace menos trabajo porque descarta un lado completo en cada paso: en promedio del orden de `n` comparaciones frente a `n log n` al ordenar todo (caso promedio; con pivotes malos puede llegar a `n²`). Si además se necesitaran varios k distintos sobre los mismos datos, convendría ordenar una sola vez.

**Ampliación (no está en el PDF):** a este método se le conoce como *quickselect*.

### Versión interactiva

La función siguiente pide listas por teclado. **Termina cuando se deja la entrada vacía** (o si `input` devuelve `None`, o si se interrumpe la entrada). Para poder ejecutarla sin teclado, recibe la función de lectura como parámetro y se prueba con un lector simulado.

In [19]:
def lector_simulado(respuestas):
    """Devuelve una función que imita a input() y, al agotar las respuestas, devuelve cadena vacía (cancelar)."""
    cola = list(respuestas)
    def lector(indicacion=""):
        if cola:
            valor = cola.pop(0)
            print(f"{indicacion}{valor}")
            return valor
        return ""
    return lector

def sesion_k_mayor(lector=input):
    """Pide una lista y un k y muestra el k-ésimo mayor. Termina con entrada vacía o None."""
    while True:
        try:
            entrada = lector("Números separados por espacios (vacío para salir): ")
        except (EOFError, KeyboardInterrupt):
            entrada = None
        if entrada is None or not entrada.strip():
            print("Sesión terminada.")
            return
        try:
            numeros = [int(texto) for texto in entrada.split()]
        except ValueError:
            print("Entrada no válida: use solo números enteros.")
            continue
        try:
            texto_k = lector("Valor de k (vacío para cancelar): ")
        except (EOFError, KeyboardInterrupt):
            texto_k = None
        if texto_k is None or not texto_k.strip():
            print("Sesión terminada.")
            return
        try:
            k = int(texto_k)
            print(f"El {k}-ésimo mayor de {numeros} es {kmayor_particion(numeros, k)}")
        except ValueError as error:
            print("Error:", error)

# Versión no interactiva: se simulan las respuestas del usuario
sesion_k_mayor(lector_simulado(["15 10 12 20 25 13 22", "3", "uno dos", "4 8 1", "9", "7 7 3", "1", ""]))

# Para usarlo de forma interactiva, quita el comentario de la siguiente línea
# sesion_k_mayor()

Números separados por espacios (vacío para salir): 15 10 12 20 25 13 22
Valor de k (vacío para cancelar): 3
El 3-ésimo mayor de [15, 10, 12, 20, 25, 13, 22] es 20
Números separados por espacios (vacío para salir): uno dos
Entrada no válida: use solo números enteros.
Números separados por espacios (vacío para salir): 4 8 1
Valor de k (vacío para cancelar): 9
Error: k debe estar entre 1 y el tamaño de la lista
Números separados por espacios (vacío para salir): 7 7 3
Valor de k (vacío para cancelar): 1
El 1-ésimo mayor de [7, 7, 3] es 7
Números separados por espacios (vacío para salir): 
Sesión terminada.


### Programación en pareja (págs. 151–153)

Las páginas 151 y 152 no traen código: son una guía para el docente sobre cómo organizar la programación en pareja. Resumen:

- Establecer pautas y mecanismos: por ejemplo, que los estudiantes **se turnen** para manejar el ratón, y prever un plan de contingencia si un integrante falta (el estudiante activo no debe ser penalizado).
- Emparejar a personas con habilidades **similares**, no necesariamente iguales; evitar juntar a estudiantes muy débiles con muy fuertes.
- La programación en pareja **no** es "divide y vencerás" (repartirse partes del problema): es un esfuerzo de colaboración real en todo el proyecto.
- Ofrecer incentivos y pedir el emparejamiento solo en una o dos tareas.
- Evitar trampas: revisar el diseño y la evaluación del curso y explicar de forma concreta qué comportamientos se consideran trampa; las tareas deben ser significativas.
- Un entorno de aprendizaje colaborativo existe siempre que se exige trabajo conjunto, con o sin evaluación directa.

El ejercicio práctico es el Q1 de la pág. 153 (el proyecto de arriba), para resolver en pareja con los dos métodos.

**Referencia:** PDF parte 1, págs. 150–153

## Práctica

Los ejercicios usan las funciones definidas arriba (`particion_primero` se define en la siguiente celda de apoyo, con la corrección del error de `partition1`). Cada solución está plegada.

In [20]:
def particion_primero(S, low, high):
    """Partición silenciosa con primer pivote (misma lógica que partition1 del PDF, sin print)."""
    return particionar(S, low, high, elegir_primero, Contador())

S = [15, 10, 12, 20, 25, 13, 22]
print(particion_primero(S, 0, len(S) - 1), S)

3 [13, 10, 12, 15, 25, 20, 22]


### Ejercicio 1 — Dividir respecto al pivote

Escribe `dividir_por_pivote(lista)` que tome el primer elemento como pivote y devuelva una tupla `(menores, pivote, mayores)`. `menores` contiene los elementos del resto estrictamente menores que el pivote y `mayores` los demás (mayores o iguales), conservando el orden original. Sin modificar la lista.

In [21]:
def dividir_por_pivote(lista):
    # Escribe tu solución aquí
    pass

In [22]:
verificar("lista del PDF", dividir_por_pivote([15, 10, 12, 20, 25, 13, 22]), ([10, 12, 13], 15, [20, 25, 22]))
verificar("un solo elemento", dividir_por_pivote([5]), ([], 5, []))
verificar("pivote máximo", dividir_por_pivote([3, 1, 2]), ([1, 2], 3, []))

Incorrecto: lista del PDF. Se esperaba ([10, 12, 13], 15, [20, 25, 22]) y se obtuvo None
Incorrecto: un solo elemento. Se esperaba ([], 5, []) y se obtuvo None
Incorrecto: pivote máximo. Se esperaba ([1, 2], 3, []) y se obtuvo None


<details><summary>Ver solución</summary>

```python
def dividir_por_pivote(lista):
    pivote = lista[0]
    resto = lista[1:]
    menores = [x for x in resto if x < pivote]
    mayores = [x for x in resto if x >= pivote]
    return menores, pivote, mayores
```

Es el paso de división del ordenamiento rápido sin intercambios: dos recorridos del resto de la lista, uno por cada lado.
</details>

### Ejercicio 2 — Posición final del pivote

Escribe `posicion_final_del_pivote(lista)` que devuelva el índice en el que quedará el primer elemento tras la partición con primer pivote, **sin** ejecutar la partición. Supón que los valores son distintos.

In [23]:
def posicion_final_del_pivote(lista):
    # Escribe tu solución aquí
    pass

In [24]:
verificar("lista del PDF", posicion_final_del_pivote([15, 10, 12, 20, 25, 13, 22]), 3)
verificar("lista ordenada", posicion_final_del_pivote([10, 12, 13, 15]), 0)
verificar("lista descendente", posicion_final_del_pivote([25, 22, 20, 15, 13, 12, 10]), 6)

Incorrecto: lista del PDF. Se esperaba 3 y se obtuvo None
Incorrecto: lista ordenada. Se esperaba 0 y se obtuvo None
Incorrecto: lista descendente. Se esperaba 6 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def posicion_final_del_pivote(lista):
    pivote = lista[0]
    return sum(1 for x in lista[1:] if x < pivote)
```

El pivote termina con tantos elementos a su izquierda como elementos menores que él existan; por eso su posición es esa cantidad.
</details>

### Ejercicio 3 — Ordenamiento rápido con listas nuevas

Escribe `quicksort_listas(lista)` que devuelva una **lista nueva** ordenada, de forma recursiva, con el primer elemento como pivote. Debe funcionar con listas vacías y con valores repetidos.

In [25]:
def quicksort_listas(lista):
    # Escribe tu solución aquí
    pass

In [26]:
verificar("lista del PDF", quicksort_listas([15, 10, 12, 20, 25, 13, 22]), [10, 12, 13, 15, 20, 22, 25])
verificar("lista vacía", quicksort_listas([]), [])
verificar("valores repetidos", quicksort_listas([3, 1, 3, 2, 3]), [1, 2, 3, 3, 3])

Incorrecto: lista del PDF. Se esperaba [10, 12, 13, 15, 20, 22, 25] y se obtuvo None
Incorrecto: lista vacía. Se esperaba [] y se obtuvo None
Incorrecto: valores repetidos. Se esperaba [1, 2, 3, 3, 3] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def quicksort_listas(lista):
    if len(lista) <= 1:
        return list(lista)
    pivote = lista[0]
    menores = [x for x in lista[1:] if x < pivote]
    iguales = [x for x in lista if x == pivote]
    mayores = [x for x in lista[1:] if x > pivote]
    return quicksort_listas(menores) + iguales + quicksort_listas(mayores)
```

El caso base son las listas de 0 o 1 elementos. Agrupar los valores iguales al pivote evita recursión innecesaria con repetidos.
</details>

### Ejercicio 4 — Partición con el último elemento como pivote

Escribe `particion_ultimo(S, low, high)` que use `S[high]` como pivote, modifique `S` in situ de modo que los menores o iguales queden a la izquierda y los mayores a la derecha, coloque el pivote en su posición final y devuelva esa posición. Esquema sugerido: recorrer `j` de `low` a `high - 1`, llevando un índice `i` que marca el final de la zona de menores o iguales.

In [27]:
def particion_ultimo(S, low, high):
    # Escribe tu solución aquí
    pass

In [28]:
S = [15, 10, 12, 20, 25, 13, 22]
pp = particion_ultimo(S, 0, 6)
verificar("posición del pivote 22", pp, 5)
verificar("izquierda menor o igual", isinstance(pp, int) and all(x <= S[pp] for x in S[:pp]), True)
verificar("derecha mayor", isinstance(pp, int) and all(x > S[pp] for x in S[pp + 1:]), True)
T = [3, 1, 2]
verificar("lista de tres", particion_ultimo(T, 0, 2), 1)
verificar("un solo elemento", particion_ultimo([5], 0, 0), 0)

Incorrecto: posición del pivote 22. Se esperaba 5 y se obtuvo None
Incorrecto: izquierda menor o igual. Se esperaba True y se obtuvo False
Incorrecto: derecha mayor. Se esperaba True y se obtuvo False
Incorrecto: lista de tres. Se esperaba 1 y se obtuvo None
Incorrecto: un solo elemento. Se esperaba 0 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def particion_ultimo(S, low, high):
    pivote = S[high]
    i = low - 1
    for j in range(low, high):
        if S[j] <= pivote:
            i += 1
            S[i], S[j] = S[j], S[i]
    S[i + 1], S[high] = S[high], S[i + 1]
    return i + 1
```

`i + 1` es la primera posición de los mayores; al intercambiar el pivote con ella queda entre ambas zonas. Es una variante de la partición del PDF (esquema de Lomuto).
</details>

### Ejercicio 5 — Mediana de tres

Escribe `mediana_de_tres(S, low, high)` que devuelva el **índice** del valor mediano entre `S[low]`, `S[medio]` y `S[high]`, con `medio = (low + high) // 2`. Es la estrategia de pivote que evita el peor caso en listas ordenadas.

In [29]:
def mediana_de_tres(S, low, high):
    # Escribe tu solución aquí
    pass

In [30]:
verificar("lista del PDF", mediana_de_tres([15, 10, 12, 20, 25, 13, 22], 0, 6), 3)
verificar("descendente", mediana_de_tres([25, 22, 20, 15, 13, 12, 10], 0, 6), 3)
verificar("tres elementos", mediana_de_tres([1, 2, 3], 0, 2), 1)
verificar("mediano al inicio", mediana_de_tres([7, 1, 9], 0, 2), 0)

Incorrecto: lista del PDF. Se esperaba 3 y se obtuvo None
Incorrecto: descendente. Se esperaba 3 y se obtuvo None
Incorrecto: tres elementos. Se esperaba 1 y se obtuvo None
Incorrecto: mediano al inicio. Se esperaba 0 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def mediana_de_tres(S, low, high):
    medio = (low + high) // 2
    candidatos = sorted([(S[low], low), (S[medio], medio), (S[high], high)])
    return candidatos[1][1]
```

Se ordenan tres pares (valor, índice) y se toma el índice del central. En una lista ordenada el elemento medio es la mediana, y la partición queda equilibrada.
</details>

### Ejercicio 6 — k-ésimo menor con partición

Escribe `kmenor(lista, k)` que devuelva el k-ésimo elemento más **pequeño** (`k = 1` es el mínimo) usando `particion_primero` y siguiendo solo el lado donde está el índice `k - 1`. No modifiques la lista original.

In [31]:
def kmenor(lista, k):
    # Escribe tu solución aquí
    pass

In [32]:
datos = [15, 10, 12, 20, 25, 13, 22]
verificar("mínimo", kmenor(datos, 1), 10)
verificar("tercero", kmenor(datos, 3), 13)
verificar("máximo", kmenor(datos, 7), 25)
verificar("no modifica la lista", datos, [15, 10, 12, 20, 25, 13, 22])

Incorrecto: mínimo. Se esperaba 10 y se obtuvo None
Incorrecto: tercero. Se esperaba 13 y se obtuvo None
Incorrecto: máximo. Se esperaba 25 y se obtuvo None
Correcto: no modifica la lista


<details><summary>Ver solución</summary>

```python
def kmenor(lista, k):
    S = list(lista)
    objetivo = k - 1
    low, high = 0, len(S) - 1
    while low <= high:
        pp = particion_primero(S, low, high)
        if pp == objetivo:
            return S[pp]
        if pp < objetivo:
            low = pp + 1
        else:
            high = pp - 1
```

Es el mismo método del proyecto con el índice objetivo `k - 1` en vez de `n - k`. Se trabaja sobre una copia para no alterar la entrada.
</details>

### Ejercicio 7 — Ordenamiento rápido iterativo

Escribe `quicksort_iterativo(S)` que ordene `S` in situ y la devuelva, **sin recursión**: usa una pila (lista) con pares `(low, high)` pendientes y la función `particion_primero`.

In [33]:
def quicksort_iterativo(S):
    # Escribe tu solución aquí
    pass

In [34]:
verificar("lista del PDF", quicksort_iterativo([15, 10, 12, 20, 25, 13, 22]), [10, 12, 13, 15, 20, 22, 25])
verificar("lista vacía", quicksort_iterativo([]), [])
verificar("valores repetidos", quicksort_iterativo([3, 1, 3, 2, 3]), [1, 2, 3, 3, 3])

Incorrecto: lista del PDF. Se esperaba [10, 12, 13, 15, 20, 22, 25] y se obtuvo None
Incorrecto: lista vacía. Se esperaba [] y se obtuvo None
Incorrecto: valores repetidos. Se esperaba [1, 2, 3, 3, 3] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def quicksort_iterativo(S):
    pila = [(0, len(S) - 1)]
    while pila:
        low, high = pila.pop()
        if low < high:
            pp = particion_primero(S, low, high)
            pila.append((low, pp - 1))
            pila.append((pp + 1, high))
    return S
```

La pila reemplaza a las llamadas recursivas: cada par pendiente es un tramo por ordenar. El resultado es idéntico al de `quicksort1`.
</details>

## Apuntes para estudiar

**Clave**

- El ordenamiento rápido es divide y vencerás recursivo: elige un **pivote**, **particiona** (menores a la izquierda, mayores a la derecha) y ordena cada lado por separado.
- La partición se hace **in situ** con dos índices (`left` y `right`) y un intercambio al final: el pivote queda en su posición definitiva (`pivotpoint`).
- `quicksort1(S, low, high)` solo trabaja mientras `low < high`; el caso base son tramos de 0 o 1 elementos.
- Diferencia con la fusión: en el rápido las dos sublistas pueden tener tamaños distintos y no hay paso de combinación; no necesita lista auxiliar.
- Con primer pivote, el peor caso (lista ordenada o inversa) divide `N - 1` elementos en cada llamada.
- `quicksort2` elige el pivote al azar con `randint(low, high)` y lo intercambia con `S[low]`.
- Estrategias: primero, último, medio, aleatorio, mediana de tres; todas solo cambian qué elemento se lleva a la primera posición.
- Complejidad (ampliación): promedio y mejor `O(n log n)`; peor `O(n²)`.
- k-ésimo mayor: o se ordena, o se particiona y se sigue solo por el lado del índice `n - k` (selección rápida).

**Errores comunes**

- Llamar `quicksort1(S, 0, len(S))` en vez de `len(S) - 1`.
- Creer que la partición deja ordenadas las dos mitades.
- Usar siempre el primer elemento como pivote con datos ya ordenados.
- Pensar que el pivote aleatorio garantiza velocidad en todos los casos.
- Usar `while left < right` en `partition1` (error del PDF): deja listas mal ordenadas; debe ser `left <= right`.
- Olvidar que muchos valores repetidos degradan este esquema de partición.
- Afirmar que el rápido "no usa memoria": no usa lista auxiliar, pero sí pila de recursión.

**Para el quiz**

- Salida de `partition1` con `[15, 10, 12, 20, 25, 13, 22]`: `[13, 10, 12, 15, 25, 20, 22]`.
- Peor caso del primer pivote: lista descendente (o ascendente), `N - 1` elementos por división.
- `partition2` elige el pivote con `randint(low, high)` e intercambia `S[low]` con `S[rand]`.
- Rápido frente a fusión: tamaños desiguales de sublistas y sin memoria adicional del tamaño de la entrada.

## Autoevaluación

**1. ¿Qué es el pivote y qué logra la partición?**

<details><summary>Ver respuesta</summary>

El pivote es el elemento de referencia. La partición reordena la lista de modo que los menores queden a su izquierda y los mayores a su derecha, y el pivote queda en su posición final.
</details>

**2. ¿En qué se diferencia el ordenamiento rápido del ordenamiento por fusión?**

<details><summary>Ver respuesta</summary>

Ambos son divide y vencerás recursivos, pero el rápido divide según el pivote (las sublistas pueden ser desiguales), hace el trabajo al dividir, no tiene paso de combinación y no necesita memoria auxiliar del tamaño de la lista.
</details>

**3. ¿Por qué el primer pivote es malo con una lista ya ordenada?**

<details><summary>Ver respuesta</summary>

Porque el primer elemento es siempre el menor del tramo: una sublista queda vacía y la otra con `n - 1` elementos. Hay `N` divisiones en lugar de unas `log n`, lo que da complejidad cuadrática.
</details>

**4. ¿Qué cambia entre `quicksort1` y `quicksort2`?**

<details><summary>Ver respuesta</summary>

Solo la partición: `partition2` elige una posición aleatoria entre `low` y `high` y la intercambia con `S[low]`, de modo que ninguna entrada concreta es sistemáticamente el peor caso.
</details>

**5. ¿Por qué encontrar el k-ésimo mayor por partición suele ser más eficiente que ordenar?**

<details><summary>Ver respuesta</summary>

Porque tras cada partición se descarta un lado completo y se sigue solo por el que contiene el índice objetivo; en promedio el trabajo es del orden de `n` en lugar de `n log n` (caso promedio).
</details>

---

[Índice de la semana](../semana-04.ipynb) | [Anterior](u28-ordenamiento-por-fusion.ipynb) | [Siguiente](../clase-2-greedy-divide-y-venceras/u30-enfoque-greedy.ipynb)